# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q sentence-transformers faiss-cpu transformers torch pandas
    print('Packages installed.')


# Retrieval-Augmented Generation (RAG) for Research In-Depth

**RAG** is a technique to improve the accuracy of Large Language Models (LLMs) by giving them access to private or specific data. Instead of relying solely on what the model 'knows' from its training, we **retrieve** relevant documents and feed them to the model as context.

This is incredibly useful for qualitative researchers who want to "chat" with their interview transcripts or literature review.

**Steps:**
1.  **Embed**: Convert text documents into numbers (vectors).
2.  **Index**: Store vectors in a searchable database (FAISS).
3.  **Retrieve**: Find the most relevant documents for a user question.
4.  **Generate**: Ask an LLM to answer the question using the retrieved text.


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
from transformers import pipeline

print("Libraries loaded.")

## 1. The Knowledge Base

Let's imagine we have a small database of research notes or interview excerpts.

In [ ]:
documents = [
    "Participant 1 mentioned that the new housing policy increased their rent by 20%.",
    "Participant 2 felt that the community center was a vital hub for social interaction.",
    "The survey results indicate a strong correlation between education level and voting behavior.",
    "Local businesses reported a decline in foot traffic due to the construction on Main Street.",
    "Participant 3 argued that public transport is unreliable during weekends.",
    "The focus group reached a consensus that green spaces are essential for mental health."
]

df = pd.DataFrame(documents, columns=['text'])
print(f"Knowledge Base: {len(df)} documents.")

## 2. Embedding Documents

We use a pre-trained model to turn text into vectors. `all-MiniLM-L6-v2` is a fast and efficient model.

In [ ]:
model_emb = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings
embeddings = model_emb.encode(df['text'].tolist())

print(f"Embeddings shape: {embeddings.shape}")
# Output: (6, 384) because we have 6 docs and the vector size is 384

## 3. Creating the Index (Vector Database)

FAISS (Facebook AI Similarity Search) allows for extremely fast similarity searches.

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 distance (Euclidean)
index.add(embeddings)

print(f"Total vectors in index: {index.ntotal}")

## 4. The Retrieval Function

When a user asks a question, we embed the question and search for the nearest vectors in our index.

In [ ]:
def retrieve_documents(query, k=2):
    # Embed the query
    query_vec = model_emb.encode([query])
    
    # Search
    distances, indices = index.search(query_vec, k)
    
    results = []
    for i in range(k):
        idx = indices[0][i]
        results.append(df.iloc[idx]['text'])
    return results

# Test
query = "How do people feel about parks?"
docs = retrieve_documents(query)
print("Top retrieved:")
for d in docs:
    print(f"- {d}")

## 5. Generation (The RAG Part)

Now we pass the retrieved context to a text generation model. We'll use `google/flan-t5-small`, which is small enough to run on a standard CPU but capable of answering questions based on context.

In [ ]:
generator = pipeline("text2text-generation", model="google/flan-t5-small")

def rag_answer(question):
    # 1. Retrieve
    context_docs = retrieve_documents(question, k=2)
    context_str = " ".join(context_docs)
    
    # 2. Prepare Prompt (Flan-T5 likes specific instructions)
    prompt = f"Answer the question based on the context.\nContext: {context_str}\nQuestion: {question}"
    
    # 3. Generate
    output = generator(prompt, max_length=100)
    return output[0]['generated_text']

# Ask a question
q1 = "What are the issues with public transport?"
print(f"Q: {q1}")
print(f"A: {rag_answer(q1)}")

print("-" * 20)

q2 = "Why is voting behavior related to education?"
print(f"Q: {q2}")
print(f"A: {rag_answer(q2)}")